# Model 4: Armington substitution

In [1]:
import numpy as np
from scipy.optimize import root
import pandas as pd

## Functions

### Production

In [2]:
def output(A, L):
    return A * L

### Prices

In [3]:
def price_factory_gate(w, A):
    return w / A

def price_ij(t, w, A):
    p_factory = price_factory_gate(w, A)
    return p_factory[:, np.newaxis] * (1 + t)

### Household

In [4]:
def utility(beta, Z):
    return np.sum(beta * np.log(Z), axis=0)

def budget_constraint(p, Z, I):
    return np.sum(p * Z, axis=0) - I

def income(w, L_bar):
    return w * L_bar

def armington_consumption_composite(beta, Z, eta):
    return (np.sum((beta ** (1 / eta)) * (Z ** ((eta - 1) / eta)), axis=0)) ** (eta / (eta - 1))

def unit_expenditure_function(beta, p, eta):
    return (np.sum(beta * (p ** (1 - eta)), axis=0)) ** (1 / (1 - eta))

def expenditure_share(beta, p, eta):
    return (beta * (p ** (1 - eta))) / (np.sum(beta * (p ** (1 - eta)), axis=0))

def demand_ij(s, I, p):
    return (s * I) / p

### Total demand

In [5]:
def output_demanded(t, Z):
    return np.sum(Z * (1 + t), axis=1)

def transport_absorbed(t, Z):
    return t * Z

### Market clearing / residuals

In [6]:
def goods_market_clearing(t, Y, Z):
    return Y - output_demanded(t, Z)

def value_market_clearing(t, w, L_bar, beta, A, eta):
    I = income(w, L_bar)
    delivered_prices = price_ij(t, w, A)
    expenditure_shares = expenditure_share(beta, delivered_prices, eta)
    Z = demand_ij(expenditure_shares, I, delivered_prices)

    revenue = np.sum(delivered_prices * Z, axis=1)
    supply = w * L_bar
    
    return revenue - supply

## Modelling values

In [7]:
L_bar = np.array([100, 100])
A = np.array([1.0, 1.0])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
t_bench = np.array([[0, 0.2], [0.2, 0]])
eta = 4

## Benchmark

In [8]:
def system_residuals(w2):
    w = np.array([1.0, w2[0]])
    value_clearing = value_market_clearing(t_bench, w, L_bar, beta, A, eta)
    return value_clearing[1]

### Value-market clearing

In [9]:
sol = root(system_residuals, [2])
print("Solution for w_2:", sol.x[0])

Solution for w_2: 1.0


In [10]:
system_residuals(sol.x)

0.0

In [11]:
w_sol = np.array([1.0, sol.x[0]])
p_sol = price_ij(t_bench, w_sol, A)
I_sol = income(w_sol, L_bar)
Y_sol = output(A, L_bar)
P_sol = unit_expenditure_function(beta, p_sol, eta)
s_sol = expenditure_share(beta, p_sol, eta)
Z_sol = demand_ij(s_sol, I_sol, p_sol)
Q_sol = armington_consumption_composite(beta, Z_sol, eta)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Demands:", Z_sol)
print("Composite price index:", P_sol)
print("Expenditure shares:", s_sol)
print("Armington consumption composite:", Q_sol)

Prices: [[1.  1.2]
 [1.2 1. ]]
Wages: [1. 1.]
Demands: [[72.16035635 23.19970304]
 [23.19970304 72.16035635]]
Composite price index: [1.06344689 1.06344689]
Expenditure shares: [[0.72160356 0.27839644]
 [0.27839644 0.72160356]]
Armington consumption composite: [94.03384482 94.03384482]


### Goods market clearing

In [12]:
goods_res = goods_market_clearing(t_bench, Y_sol, Z_sol)
print("Goods market residuals:", goods_res)

Goods market residuals: [0. 0.]


## Experiments

### Transport costs

In [13]:
eta = 4
t_vals = [0, 0.05, 0.1, 0.2, 0.4]
results = []

for t in t_vals:
    t_matrix = np.array([[0, t], [t, 0]])

    def temp_residuals(w2):
        w = np.array([1.0, w2[0]])
        value_clearing = value_market_clearing(t_matrix, w, L_bar, beta, A, eta)
        return value_clearing[1]

    sol = root(temp_residuals, [2])
    w_sol = np.array([1.0, sol.x[0]])

    I_sol = income(w_sol, L_bar)
    p_sol = price_ij(t_matrix, w_sol, A)
    s_sol = expenditure_share(beta, p_sol, eta)
    Z_sol = demand_ij(s_sol, I_sol, p_sol)
    Q_sol = armington_consumption_composite(beta, Z_sol, eta)
    P_sol = unit_expenditure_function(beta, p_sol, eta)

    results.append({
        't': t,
        's12': s_sol[0, 1],
        's21': s_sol[1, 0],
        'Z12': Z_sol[0, 1],
        'Z21': Z_sol[1, 0],
        'P1': P_sol[0],
        'P2': P_sol[1],
        'Q1': Q_sol[0],
        'Q2': Q_sol[1]
    })

In [14]:
results_df = pd.DataFrame(results)

In [15]:
from lets_plot import *
LetsPlot.setup_html()

p1 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='s21')) + labs(y = r'\(s_{21}\)', title = "Import expenditure share in node 1") + \
      theme_bw()
p2 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='s12')) + labs(y = r'\(s_{12}\)', title = "Import expenditure share in node 2") + \
      theme_bw()
p3 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z21')) + labs(y = r'\(Z_{21}\)', title = "Import quantities in node 1") + \
      theme_bw()
p4 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z12')) + labs(y = r'\(Z_{12}\)', title = "Import quantities in node 2") + \
      theme_bw()
p5 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='P1')) + labs(y = r'\(P_1\)', title = "Composite price index in node 1") + \
      theme_bw()
p6 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='P2')) + labs(y = r'\(P_2\)', title = "Composite price index in node 2") + \
      theme_bw()
p7 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q1')) + labs(y = r'\(Q_1\)', title = "Real consumption in node 1") + \
      theme_bw()
p8 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q2')) + labs(y = r'\(Q_2\)', title = "Real consumption in node 2") + \
      theme_bw()

grid = gggrid([p1, p2, p3, p4, p5, p6, p7, p8], ncol=2)

grid

ggsave(grid, "fig/comparison_transport_costs_4.svg")

'/Users/timnehls/Library/CloudStorage/OneDrive-Persönlich/uni/umucourses/MMES/lab2/lets-plot-images/fig/comparison_transport_costs_4.svg'

### Substitution elasticity

In [16]:
eta_vals = [1.5, 2, 4, 8]
t_vals = [0, 0.05, 0.1, 0.2, 0.4]
results = []

for eta in eta_vals:
    for t in t_vals:
        t_matrix = np.array([[0, t], [t, 0]])

        def temp_residuals(w2):
            w = np.array([1.0, w2[0]])
            value_clearing = value_market_clearing(t_matrix, w, L_bar, beta, A, eta)
            return value_clearing[1]

        sol = root(temp_residuals, [2])
        w_sol = np.array([1.0, sol.x[0]])

        I_sol = income(w_sol, L_bar)
        p_sol = price_ij(t_matrix, w_sol, A)
        s_sol = expenditure_share(beta, p_sol, eta)
        Z_sol = demand_ij(s_sol, I_sol, p_sol)
        Q_sol = armington_consumption_composite(beta, Z_sol, eta)
        
        P_sol = unit_expenditure_function(beta, p_sol, eta)

        results.append({
            't': t,
            'eta': eta,
            's12': s_sol[0, 1],
            's21': s_sol[1, 0],
            'Z12': Z_sol[0, 1],
            'Z21': Z_sol[1, 0],
            'P1': P_sol[0],
            'P2': P_sol[1],
            'Q1': Q_sol[0],
            'Q2': Q_sol[1]
        })

In [17]:
results_df = pd.DataFrame(results)

results_df["eta"] = results_df["eta"].astype("category")

In [18]:
from lets_plot import *
LetsPlot.setup_html()

p1 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='s21', color=as_discrete('eta'))) + \
      labs(y = r'\(s_{21}\)', title = "Import expenditure share in node 1", color = r'\(\eta\)') + \
      theme_bw()
p2 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='s12', color=as_discrete('eta'))) + \
      labs(y = r'\(s_{12}\)', title = "Import expenditure share in node 2", color = r'\(\eta\)') + \
      theme_bw()
p3 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z21', color=as_discrete('eta'))) + \
      labs(y = r'\(Z_{21}\)', title = "Import quantities in node 1", color = r'\(\eta\)') + \
      theme_bw()
p4 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z12', color=as_discrete('eta'))) + \
      labs(y = r'\(Z_{12}\)', title = "Import quantities in node 2", color = r'\(\eta\)') + \
      theme_bw()
p5 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='P1', color=as_discrete('eta'))) + \
      labs(y = r'\(P_1\)', title = "Composite price index in node 1", color = r'\(\eta\)') + \
      theme_bw()
p6 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='P2', color=as_discrete('eta'))) + \
      labs(y = r'\(P_2\)', title = "Composite price index in node 2", color = r'\(\eta\)') + \
      theme_bw()
p7 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q1', color=as_discrete('eta'))) + \
      labs(y = r'\(Q_1\)', title = "Real consumption in node 1", color = r'\(\eta\)') + \
      theme_bw()
p8 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q2', color=as_discrete('eta'))) + \
      labs(y = r'\(Q_2\)', title = "Real consumption in node 2", color = r'\(\eta\)') + \
      theme_bw()

grid = gggrid([p1, p2, p3, p4, p5, p6, p7, p8], ncol=2)

grid

ggsave(grid, "fig/comparison_elasticities_4.svg")

'/Users/timnehls/Library/CloudStorage/OneDrive-Persönlich/uni/umucourses/MMES/lab2/lets-plot-images/fig/comparison_elasticities_4.svg'

### Productivity asymmetry

In [19]:
A1_vals = [0.8, 1, 1.2, 1.5]
A2 = 1
results = []
eta = 4

for A1 in A1_vals:
    A = np.array([A1, A2])

    def temp_residuals(w2):
        w = np.array([1.0, w2[0]])
        value_clearing = value_market_clearing(t_bench, w, L_bar, beta, A, eta)
        return value_clearing[1]

    sol = root(temp_residuals, [2])
    w_sol = np.array([1.0, sol.x[0]])

    I_sol = income(w_sol, L_bar)
    p_sol = price_ij(t_bench, w_sol, A)
    s_sol = expenditure_share(beta, p_sol, eta)
    Z_sol = demand_ij(s_sol, I_sol, p_sol)
    Q_sol = armington_consumption_composite(beta, Z_sol, eta)
    
    P_sol = unit_expenditure_function(beta, p_sol, eta)
    Y_sol = output(A, L_bar)

    results.append({
        'A1': A1,
        'w1': w_sol[0],
        'w2': w_sol[1],
        'w2/w1': w_sol[1] / w_sol[0],
        'p1': p_sol[0, 0],
        'p2': p_sol[1, 1],
        'p1/p2': p_sol[0, 0] / p_sol[1, 1],
        's11': s_sol[0, 0],
        's12': s_sol[0, 1],
        's21': s_sol[1, 0],
        's22': s_sol[1, 1],
        'Q1': Q_sol[0],
        'Q2': Q_sol[1],
        'P1': P_sol[0],
        'P2': P_sol[1],
        'Y1': Y_sol[0],
        'Y2': Y_sol[1]
    })

In [20]:
results_df = pd.DataFrame(results)

We expect that if A1 is below A2 (and below its previous value 1.0), production is more labour intensive, so the expenditure on wages per unit is higher than before for firm 1. As w1 is fixed, w2 will increase (because the labour is more productive in firm 2). Due to the lower productivity, goods from firm 1 will become relatively more expensive than goods from firm 2, and the import expenditure share in node 1 will be higher (as goods are substituted with those from firm 2), and the local expenditure share in node 2 will be higher as well. The real consumption in both nodes will then be lower than for A1 = A2 = 1, as there is a loss of productivity in total. As wage 1 is fixed, the income in node 1 will be lower than in node 2, thus, also real consumption will be lower (due to the higher prices of products).

In [21]:
results_df

,A1,w1,w2,w2/w1,p1,p2,p1/p2,s11,s12,s21,s22,Q1,Q2,P1,P2,Y1,Y2
0,0.8,1.0,1.198732,1.198732,1.250000,1.198732,1.042769,0.695676,0.253872,0.304324,0.746128,76.150262,92.992069,1.313193,1.289069,80.0,100.0
1,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.721604,0.278396,0.278396,0.721604,94.033845,94.033845,1.063447,1.063447,100.0,100.0
2,1.2,1.0,0.862339,0.862339,0.833333,0.862339,0.966364,0.741748,0.299478,0.258252,0.700522,111.809700,94.967828,0.894377,0.908033,120.0,100.0
3,1.5,1.0,0.719412,0.719412,0.666667,0.719412,0.926683,0.765103,0.326513,0.234897,0.673487,138.325351,96.221924,0.722933,0.747659,150.0,100.0


In [23]:
from lets_plot import *
LetsPlot.setup_html()

p1 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='w2/w1')) + labs(y = r'\(w_2/w_1\)', title = "Relative wage in node 2", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p2 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='p1/p2')) + labs(y = r'\(p_1/p_2\)', title = "Relative prices", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p3 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='s11')) + labs(y = r'\(s_{11}\)', title = "Domestic expenditure share in node 1", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p4 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='s12')) + labs(y = r'\(s_{12}\)', title = "Import expenditure share in node 2", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p5 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='s21')) + labs(y = r'\(s_{21}\)', title = "Import expenditure share in node 1", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p6 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='s22')) + labs(y = r'\(s_{22}\)', title = "Domestic expenditure share in node 2", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p7 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='Q1')) + labs(y = r'\(Q_1\)', title = "Real consumption in node 1", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()
p8 = ggplot(results_df, aes(x='A1')) + \
      geom_line(aes(y='Q2')) + labs(y = r'\(Q_2\)', title = "Real consumption in node 2", x = r'\(A_1\)') + \
      geom_vline(xintercept = 1, linetype="dashed", color="grey") + \
      theme_bw()

grid = gggrid([p1, p2, p3, p6, p5, p4, p7, p8], ncol=2)

grid

ggsave(grid, "fig/comparison_productivity_4.svg")

'/Users/timnehls/Library/CloudStorage/OneDrive-Persönlich/uni/umucourses/MMES/lab2/lets-plot-images/fig/comparison_productivity_4.svg'

## Part VI: Comparison of models III–IV

In [90]:
L_bar = np.array([100, 100])
A = np.array([1.0, 1.0])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
t_bench = np.array([[0, 0.1], [0.1, 0]])
eta = 4

In [91]:
def system_residuals(w2):
    w = np.array([1.0, w2[0]])
    value_clearing = value_market_clearing(t_bench, w, L_bar, beta, A, eta)
    return value_clearing[1]

In [92]:
sol = root(system_residuals, [2])
print("Solution for w_2:", sol.x[0])

Solution for w_2: 1.0


In [93]:
system_residuals(sol.x)

0.0

In [95]:
w_sol = np.array([1.0, sol.x[0]])
p_sol = price_ij(t_bench, w_sol, A)
I_sol = income(w_sol, L_bar)
Y_sol = output(A, L_bar)
P_sol = unit_expenditure_function(beta, p_sol, eta)
s_sol = expenditure_share(beta, p_sol, eta)
Z_sol = demand_ij(s_sol, I_sol, p_sol)
Q_sol = armington_consumption_composite(beta, Z_sol, eta)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Demands:", Z_sol)
print("Composite price index:", P_sol)
print("Expenditure shares:", s_sol)
print("Armington consumption composite:", Q_sol)
print("Total output:", Y_sol)

Prices: [[1.  1.1]
 [1.1 1. ]]
Wages: [1. 1.]
Demands: [[66.62773235 30.33842513]
 [30.33842513 66.62773235]]
Composite price index: [1.0355425 1.0355425]
Expenditure shares: [[0.66627732 0.33372268]
 [0.33372268 0.66627732]]
Armington consumption composite: [96.56774111 96.56774111]
Total output: [100. 100.]


In [ ]:
goods_res = goods_market_clearing(t_bench, Y_sol, Z_sol)
print("Goods market residuals:", goods_res)

Goods market residuals: [0. 0.]
